# Define variables

In [1]:
import pathlib
import os
import subprocess

watermark_path = pathlib.Path("/home/ubuntu/drives/mkt/2 - Content/2 - Graphics/Animations/Logo final.png")
root_path_dir = pathlib.Path(f"/home/ubuntu/Documents/animations_059_Generalli/building_131/")
images_case = "ux_y"

# images_case = "animation_vortex"

# watermark_position = (30,30)
# watermark_position = (30, 690)
watermark_position = (1250,720)
watermark_scale = .2

crop_margins = (0, 0, 0, 0)

gif_size_list = [1920, 1080]
speed_multiplier_list = [1.0]
framerate_default = 15

test_single_image = False
add_watermark_to_images = True
turn_into_video = True
generate_intermediate_frames = True
target_framerate = 24


# Adds watermarks

In [2]:
from PIL import Image
import pathlib

def add_watermark_to_image(
    image: Image, watermark: Image, watermark_position: tuple, 
) -> Image:
    image.paste(watermark, watermark_position, watermark)
    return image

def rescale_image(image: Image, image_scale: float) -> Image:
    new_width = int(image.width*image_scale)
    new_height = int(image.height*image_scale)
    image = image.resize((new_width, new_height), Image.Resampling.LANCZOS)
    return image

def crop_image_margins(
    image: pathlib.Path, crop_margins: tuple
) -> Image:
    w, h =  image.width, image.height
    image = image.crop((crop_margins[0], crop_margins[1], w-crop_margins[2], h-crop_margins[3]))
    return image

In [3]:
def process_image(image: Image, crop_margins: tuple, watermark_position: tuple=None, watermark: Image=None) -> Image:
    image = crop_image_margins(image=image, crop_margins=crop_margins)
    image = add_watermark_to_image(image=image, watermark=watermark, watermark_position=watermark_position)
    return image

In [4]:
def generate_mp4_from_images(images_dir: pathlib.Path, video_dir: pathlib.Path, image_size: int, speed_multiplier: float):
    mp4_file = video_dir/f"{images_dir.stem}_size{image_size}_speed{speed_multiplier}.mp4"
    mp4_file.unlink(missing_ok=True)
    command = f'ffmpeg -framerate {framerate_default} -pattern_type glob -i "{images_dir}/*.png" \
    -filter_complex "setpts={1/speed_multiplier}*PTS,scale={image_size}:-2:flags=lanczos" \
    -c:v libx264 -preset slow -crf 24 -pix_fmt yuv420p "{mp4_file}"'
    print("command:", command)
    try:
        print(os.getcwd())
        subprocess.run(command, shell=True, check=True)
    except subprocess.CalledProcessError as e:
        print(f"Error: {e}")
    return mp4_file

In [5]:
def generate_frames_for_video(video_file: pathlib.Path):
    output_path = video_file.parent/f"{video_file.stem}_interpolated.mp4"
    command = f'ffmpeg -y -i "{video_file}" -filter_complex "[0:v]minterpolate=mi_mode=mci:mc_mode=none:me_mode=bidir:fps={target_framerate}[v];[v]setpts=1*PTS" "{output_path}"'
    print("command:", command)
    try:
        print(os.getcwd())
        subprocess.run(command, shell=True, check=True)
    except subprocess.CalledProcessError as e:
        print(f"Error: {e}")

In [6]:
path_use_root = root_path_dir / images_case

images_path_dir = path_use_root / "raw"
images_with_watermark_path_dir = path_use_root/ f"watermarked/"
videos_path = path_use_root/ f"videos/"

images_with_watermark_path_dir.mkdir(parents=True, exist_ok=True)
videos_path.mkdir(parents=True, exist_ok=True)


# sorted([im for im in images_path_dir.iterdir()])

In [7]:
if test_single_image:
    sample_file_path = next(images_path_dir.iterdir())
    sample_file = Image.open(sample_file_path)
    watermark = Image.open(watermark_path)
    watermark = rescale_image(image=watermark, image_scale=watermark_scale)
    img = process_image(image=sample_file, crop_margins=crop_margins ,watermark_position=watermark_position, watermark=watermark)
    img.save(images_with_watermark_path_dir/sample_file_path.name)

In [8]:
if add_watermark_to_images:
    watermark = Image.open(watermark_path)
    watermark = rescale_image(image=watermark, image_scale=watermark_scale)
    for item in sorted([im for im in images_path_dir.iterdir()]):
        if item.is_dir():
            continue
        else:
            try:
                # print(item.name)
                with Image.open(item) as img:
                    img = process_image(image=img, crop_margins=crop_margins, watermark_position=watermark_position, watermark=watermark)
                    img.save(images_with_watermark_path_dir/item.name)
            except IOError:
                continue

In [9]:
if turn_into_video:
    for image_size in gif_size_list:
        for speed_multiplier in speed_multiplier_list:
            video_out = generate_mp4_from_images(
                images_dir=images_with_watermark_path_dir, 
                video_dir=videos_path,
                image_size=image_size,
                speed_multiplier=speed_multiplier,
            )


command: ffmpeg -framerate 15 -pattern_type glob -i "/home/ubuntu/Documents/animations_059_Generalli/building_131/ux_y/watermarked/*.png"     -filter_complex "setpts=1.0*PTS,scale=1920:-2:flags=lanczos"     -c:v libx264 -preset slow -crf 24 -pix_fmt yuv420p "/home/ubuntu/Documents/animations_059_Generalli/building_131/ux_y/videos/watermarked_size1920_speed1.0.mp4"
/home/ubuntu/Documents/Codigos/AeroSim/cfdmod


ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

command: ffmpeg -framerate 15 -pattern_type glob -i "/home/ubuntu/Documents/animations_059_Generalli/building_131/ux_y/watermarked/*.png"     -filter_complex "setpts=1.0*PTS,scale=1080:-2:flags=lanczos"     -c:v libx264 -preset slow -crf 24 -pix_fmt yuv420p "/home/ubuntu/Documents/animations_059_Generalli/building_131/ux_y/videos/watermarked_size1080_speed1.0.mp4"
/home/ubuntu/Documents/Codigos/AeroSim/cfdmod


ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

In [10]:
if generate_intermediate_frames:
    for image_size in gif_size_list:
        for speed_multiplier in speed_multiplier_list:
            video_out = videos_path/f"{images_with_watermark_path_dir.stem}_size{image_size}_speed{speed_multiplier}.mp4"
            generate_frames_for_video(video_file=video_out)


command: ffmpeg -y -i "/home/ubuntu/Documents/animations_059_Generalli/building_131/ux_y/videos/watermarked_size1920_speed1.0.mp4" -filter_complex "[0:v]minterpolate=mi_mode=mci:mc_mode=none:me_mode=bidir:fps=24[v];[v]setpts=1*PTS" "/home/ubuntu/Documents/animations_059_Generalli/building_131/ux_y/videos/watermarked_size1920_speed1.0_interpolated.mp4"
/home/ubuntu/Documents/Codigos/AeroSim/cfdmod


ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

command: ffmpeg -y -i "/home/ubuntu/Documents/animations_059_Generalli/building_131/ux_y/videos/watermarked_size1080_speed1.0.mp4" -filter_complex "[0:v]minterpolate=mi_mode=mci:mc_mode=none:me_mode=bidir:fps=24[v];[v]setpts=1*PTS" "/home/ubuntu/Documents/animations_059_Generalli/building_131/ux_y/videos/watermarked_size1080_speed1.0_interpolated.mp4"
/home/ubuntu/Documents/Codigos/AeroSim/cfdmod


ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena